# Segundo Reporte
**Periodo:** 16 al 25 de febrero del 2026
## Índice
1. [Objetivo del análisis](#objetivo)
2. [Simulación de Modelos e impacto muestral](#simulacion)
3. [¿Qué tranformación es mejor según la métrica AIC?](#aic)
4. [Código en Python y Resultados clave 1.0](#codigo1)
5. [Código en Python y Resultados clave 2.0](#codigo2)
6. [Estimamos el modelo original y Revisamos las correlaciones](#correlaciones)
7. [Preguntas](#preguntas)
8. [Incluimos dummies y vemos como interactúan las variables entre ellas](#dummies)
9. [Aprendizajes obtenidos](#aprendizajes)

#Objetivo del análisis <a name="objetivo"></a>
En este periodo , nos centramos en la validación de modelos de Regresión Lineal Múltiple mediante simulación y el análisis de formas funcionales óptimas utilizando la transformación Box-Cox. Además tuvimos como objetivo desarrollar un modelo econométrico eficiente para predecir el Precio por Acción de una empresa, evaluando variables financieras y estratégicas identificando problemas de multicolinealidad.

# Simulación de Modelos e impacto muestral <a name="simulacion"></a>

En la primera semana generamos un modelo simulado con 80 observaciones donde la verdadera relación era $y = 5 + 2x_1 + 3x_2 + e$, la prueba F ($119.4$, $p \approx 0$) y las pruebas T individuales ($p = 0.000$) demostraron que cuando la relación lineal es verdadera y hay suficientes datos, el modelo de mínimos cuadrados identifica el impacto real de cada variable. Además identificamos el peligro de las muestras pequeñas al agregar variables, ya que destruimos los grados de libertidad y el modelo pierda toda su precisión estadística individual.



# ¿Qué tranformación es mejor según la métrica AIC? <a name="aic"></a>

Simulamos un escenario con 200 datos donde la relación lineal era una cruva, así que no podíamos forzar un modelo lineal pues generaríamos errores graves, así que recurrimos a algunas transformaciones y nos quedamos con la de mejor ajuste.

# Código en Python y Resultados clave 1.0 <a name="codigo1"></a>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# 1. Generación de datos simulados (n=200)
np.random.seed(42)
x1 = np.random.uniform(1, 10, 200)
x2 = np.random.uniform(0.5, 5, 200)
y_true = 5 * (x1**0.6) * (x2**0.4)
y = y_true * np.exp(np.random.normal(0, 0.4, 200))
df = pd.DataFrame({"y": y, "x1": x1, "x2": x2})

X_const = sm.add_constant(df[["x1", "x2"]])

# 2. Ajuste de los 3 Modelos
# Modelo Lineal
modelo_lin = sm.OLS(df["y"], X_const).fit()

# Modelo Log-Lineal
modelo_log = sm.OLS(np.log(df["y"]), X_const).fit()

# Modelo Box-Cox
lambda_opt = stats.boxcox_normmax(df["y"].values, brack=(-2, 2))
y_bc = stats.boxcox(df["y"].values, lmbda=lambda_opt)
modelo_bc = sm.OLS(y_bc, X_const).fit()

# 3. CREACIÓN DE LA TABLA RESUMEN (Ahorro de espacio)
tabla_resultados = pd.DataFrame({
    "Modelo": ["Lineal (Original)", "Log-Lineal (ln y)", f"Box-Cox (λ = {lambda_opt:.3f})"],
    "R² Ajustado": [modelo_lin.rsquared_adj, modelo_log.rsquared_adj, modelo_bc.rsquared_adj],
    "AIC (Criterio Akaike)": [modelo_lin.aic, modelo_log.aic, modelo_bc.aic],
    "Prob F-statistic": [modelo_lin.f_pvalue, modelo_log.f_pvalue, modelo_bc.f_pvalue]
})

# Mostrar tabla redondeada para que luzca profesional
tabla_resultados.round(4)

,Modelo,R² Ajustado,AIC (Criterio Akaike),Prob F-statistic
0,Lineal (Original),0.3811,1459.3271,0.0
1,Log-Lineal (ln y),0.4899,227.1069,0.0
2,Box-Cox (λ = 0.247),0.4752,509.0364,0.0


Observamos que nuestro modelo lineal falla en capturar la relación curva de los datos al tener un $R^2$ bajo ($0.387$) y un AIC de $1459$, por lo que nuestro modelo lineal falla en capturar la relación curva de los datos. Al aplicar logaritmo a la variable dependiente ($ln(y)$), el $R^2$ sube a $0.4899$ y el AIC cae drásticamente a $227.1$, por lo que podemos decir que esta transformación es mucho más adecuada para los datos, mejorando la precisión y la normalidad de los errores, finalmente el modelo Box-Cox explica el 48% de la variación de la variable transformada lo que es una mejora notable frente al 38% del modelo lineal original, pero tiene un AIC de $509.03$ . Así que aunque la transformación Box-Cox es mejor que el modelo lineal simple, en este ejemplo en específico el modelo Log-Lineal sigue siendo el más eficiente por tener el AIC más bajo.

# Código en Python y Resultados clave 2.0 <a name="codigo2"></a>




In [ ]:

data={"PPS": [83.69, 85.34, 72.14, 81.99, 76.19, 84.72, 77.6, 75.41, 80.6, 81.89, 81.99, 85.9, 76.77, 81.78, 80.32, 78.0, 80.55, 87.41, 76.58, 81.41, 84.55, 82.62, 80.4, 83.84, 82.15, 83.19, 78.08, 78.42, 79.19, 82.22, 87.97, 81.61, 79.49, 88.17, 77.59, 75.93, 81.85, 81.39, 79.42, 86.17, 81.69, 82.83, 81.91, 76.77, 83.02, 77.51, 82.81, 80.69, 83.55, 79.12],
      "DPS": [3.52, 3.16, 2.64, 2.9, 2.2, 3.53, 2.87, 2.81, 3.07, 2.74, 2.52, 3.32, 2.45, 3.2, 2.74, 3.08, 2.59, 3.28, 2.65, 2.49, 2.68, 3.0, 2.85, 3.31, 3.25, 3.23, 2.87, 2.46, 2.4, 3.1, 3.44, 3.11, 2.73, 3.8, 3.34, 2.68, 3.45, 3.25, 2.86, 3.22, 2.63, 2.99, 3.42, 2.45, 3.16, 2.21, 2.75, 2.53, 2.93, 2.78],
      "RE": [4.05, 3.6, 3.01, 3.24, 2.58, 3.91, 3.21, 3.14, 3.39, 3.22, 2.96, 3.84, 2.87, 3.65, 3.14, 3.66, 3.09, 3.63, 3.0, 2.86, 3.09, 3.41, 3.18, 3.66, 3.62, 3.64, 3.22, 2.9, 2.74, 3.48, 3.75, 3.45, 3.24, 4.1, 3.73, 3.0, 3.79, 3.63, 3.26, 3.61, 3.01, 3.35, 3.95, 2.87, 3.66, 2.44, 3.13, 2.94, 3.31, 3.17],
      "Sales": [1298, 1614, 1798, 1199, 1662, 1349, 1273, 1416, 1396, 1570, 1599, 1366, 886, 1503, 1952, 1391, 1534, 1367, 1302, 1699, 1439, 1773, 1404, 1494, 1496, 1664, 1415, 1693, 1382, 1771, 1754, 2087, 1918, 1161, 1343, 1362, 1678, 1413, 1195, 1395, 1627, 1175, 1522, 1295, 1604, 2002, 1629, 1309, 1768, 1918],
      "Estrategia": [1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0]
      }

df = pd.DataFrame(data)
df.head()


,PPS,DPS,RE,Sales,Estrategia
0,83.69,3.52,4.05,1298,1
1,85.34,3.16,3.60,1614,0
2,72.14,2.64,3.01,1798,1
3,81.99,2.90,3.24,1199,1
4,76.19,2.20,2.58,1662,1


# Estimamos el modelo original y Revisamos las correlaciones <a name="correlaciones"></a>

In [ ]:
df.corr()

,PPS,DPS,RE,Sales,Estrategia
PPS,1.000000,0.645262,0.636819,-0.014868,0.319054
DPS,0.645262,1.000000,0.983995,-0.153047,0.244378
RE,0.636819,0.983995,1.000000,-0.155107,0.251811
Sales,-0.014868,-0.153047,-0.155107,1.000000,-0.027646
Estrategia,0.319054,0.244378,0.251811,-0.027646,1.000000




1.- ¿Cuál es la correlación entre DPS y RE? La correlación entre DPS y RE es de 0.983995.

2.- ¿La correlación sugiere multicolinealidad según los criterios del capítulo (0.8 o 0.9)?
Una correlación tan cercana a 1 indica que ambas variables contienen casi la misma información, lo cual nos señala una multicolinealidad fuerte que le quita precisión al modelo.

3.-¿Cuáles variables parecen ser más colineales?

RE-PPSPPS y DPS: 0.645262

PPS y RE: 0.636819

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

x = df[['DPS', 'RE', 'Sales']]
X = sm.add_constant(x)

vifs= pd.DataFrame()
vifs["variable"] = X.columns
vifs["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vifs

,variable,VIF
0,const,149.806781
1,DPS,31.491538
2,RE,31.512027
3,Sales,1.024657


# Preguntas: <a name="preguntas"></a>

4.¿Qué variable tiene mayor VIF?

La variable con el mayor VIF es RE omitiendo "const"

5.Según el capítulo, ¿el VIF indica problema grave (>10) o moderado?

Indica un problema grave

In [ ]:
df['lnPPS'] = np.log(df['PPS'])
df['lnSales'] = np.log(df['Sales'])

modelo_log_pps = sm.OLS(df['lnPPS'], sm.add_constant(df['lnSales'])).fit()

# Tabla compacta para Log-Log
tabla_log = pd.DataFrame({
    "Variable": ["Constante", "ln(Sales)"],
    "Coeficiente": modelo_log_pps.params.values,
    "P-value": modelo_log_pps.pvalues.values
})
print(f"R² Ajustado: {modelo_log_pps.rsquared_adj:.4f} | Prob F-statistic: {modelo_log_pps.f_pvalue:.4f}")
tabla_log.round(4)

R² Ajustado: -0.0207 | Prob F-statistic: 0.9375


,Variable,Coeficiente,P-value
0,Constante,4.3732,0.0000
1,ln(Sales),0.0029,0.9375


6.- Interpreta $\beta$ como elasticidad.


$\beta$ nos dice que en promedio y manteniendo todo lo demás constante, un incremento del 1% en las ventas (Sales) está asociado con un aumento marginal del 0.0029% en el precio por acción (PPS).

7.- ¿La relación log-log parece linealizable? No, por su valor tan alto en el p-value, además la $R^2$ es cero, lo que significa que transformar las variables no ayudó a encontrar una relación lineal válida entre ellas.

8.-¿Qué ventaja tiene una tranformación con respecto al modelo original? Podemos interpretar los coeficientes directamente como cambios porcentuales y podemos reducir problemas de heterocedasticidad.

# Incluimos dummies y vemos como interactúan las variables entre ellas. <a name="dummies"></a>



Modelo 1:

$$ PPS = \alpha + \beta Sales + \gamma D + \delta (Sales * D) + \epsilon $$

In [ ]:
df['Sales_D'] = df['Sales']* df['Estrategia']
x2 = df[['Sales', 'Estrategia', 'Sales_D']]
X2 = sm.add_constant(x2)

modelo_int = sm.OLS(df['PPS'],X2).fit()
modelo_int.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    PPS   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.133
Method:                 Least Squares   F-statistic:                     3.504
Date:                Sun, 01 Mar 2026   Prob (F-statistic):             0.0226
Time:                        04:43:14   Log-Likelihood:                -126.72
No. Observations:                  50   AIC:                             261.4
Df Residuals:                      46   BIC:                             269.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         74.2332      3.974     18.681      0.000      66.234      82.232
Sales          0.0038      0.003      1.473      0.148      -0.001       0.009
Estrategia    14.4955      5.728      2.531      0.015       2.966      26.026
Sales_D       -0.0081      0.004     -2.181      0.034      -0.016      -0.001
==============================================================================
Omnibus:                        2.012   Durbin-Watson:                   2.369
Prob(Omnibus):                  0.366   Jarque-Bera (JB):                1.211
Skew:                          -0.060   Prob(JB):                        0.546
Kurtosis:                       3.753   Cond. No.                     2.53e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.53e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

9. Interpreta $\gamma$: ¿cómo cambia el nivel del PPS cuando Estrategia = 1? Si la empresa utiliza la Estrategia 1, su nivel base de Precio por Acción (el intercepto cuando las ventas son cero) aumenta en 14.4955 unidades en comparación con las empresas que usan la Estrategia 0.
10. Interpreta $\delta$: ¿cambia la pendiente?¿Cuánto? Sí, la pendiente cambia, para la estrategia 0, la pendiente de las ventas es $0.0038$ y para la estrategia 1, la pendiente disminuye en $0.0081$, por lo que la nueva pendiente es $0.0038 - 0.0081 = -0.0043$.
11. ¿Cuál estrategia parece más sensible a cambios en ventas?
La estrategia 1 es ligeramente más sensible a las ventas, aunque el impacto es inverso porque ante mayores ventas, el PPS tiende a caer.
12. Genera un modelo eficiente en las pruebas individuales y en la prueba global y agrega su interpretación.


Modelo 2
$$ PPS = \alpha  + \gamma D + \delta (Sales * D) + \epsilon $$

In [ ]:
x2 = df[['Estrategia', 'Sales_D']]
X2 = sm.add_constant(x2)

modelo_2 = sm.OLS(df['PPS'], X2).fit()
modelo_2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    PPS   R-squared:                       0.148
Model:                            OLS   Adj. R-squared:                  0.111
Method:                 Least Squares   F-statistic:                     4.071
Date:                Sun, 01 Mar 2026   Prob (F-statistic):             0.0234
Time:                        04:43:14   Log-Likelihood:                -127.87
No. Observations:                  50   AIC:                             261.7
Df Residuals:                      47   BIC:                             267.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         80.0096      0.644    124.225      0.000      78.714      81.305
Estrategia     8.7191      4.226      2.063      0.045       0.218      17.220
Sales_D       -0.0043      0.003     -1.590      0.119      -0.010       0.001
==============================================================================
Omnibus:                        0.956   Durbin-Watson:                   2.401
Prob(Omnibus):                  0.620   Jarque-Bera (JB):                0.336
Skew:                          -0.119   Prob(JB):                        0.846
Kurtosis:                       3.323   Cond. No.                     1.00e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large,  1e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Modelo 3: $$ PPS = \alpha + \gamma D + \delta(Sales * D) + \theta RE + \epsilon $$

In [ ]:
x3 = df[['Estrategia', 'Sales_D', 'RE']]
X3 = sm.add_constant(x3)
modelo_iii = sm.OLS(df['PPS'], X3).fit()
modelo_iii.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    PPS   R-squared:                       0.444
Model:                            OLS   Adj. R-squared:                  0.408
Method:                 Least Squares   F-statistic:                     12.25
Date:                Sun, 01 Mar 2026   Prob (F-statistic):           5.14e-06
Time:                        04:43:14   Log-Likelihood:                -117.19
No. Observations:                  50   AIC:                             242.4
Df Residuals:                      46   BIC:                             250.0
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         63.5186      3.371     18.842      0.000      56.733      70.304
Estrategia     4.5477      3.551      1.281      0.207      -2.600      11.696
Sales_D       -0.0022      0.002     -0.981      0.332      -0.007       0.002
RE             5.1024      1.030      4.952      0.000       3.029       7.176
==============================================================================
Omnibus:                        3.076   Durbin-Watson:                   1.959
Prob(Omnibus):                  0.215   Jarque-Bera (JB):                2.079
Skew:                          -0.439   Prob(JB):                        0.354
Kurtosis:                       3.475   Cond. No.                     1.13e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.13e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

$$PPS = \alpha + \beta(DPS) + \theta(Sales) + \gamma(D) + \delta(Sales \times D) + e$$

In [ ]:
x4= df[['Estrategia', 'RE']]
X4= sm.add_constant(x4)

modelo_4 = sm.OLS(df['PPS'], X4).fit()
modelo_4.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    PPS   R-squared:                       0.432
Model:                            OLS   Adj. R-squared:                  0.408
Method:                 Least Squares   F-statistic:                     17.90
Date:                Sun, 01 Mar 2026   Prob (F-statistic):           1.66e-06
Time:                        04:43:14   Log-Likelihood:                -117.71
No. Observations:                  50   AIC:                             241.4
Df Residuals:                      47   BIC:                             247.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         62.9039      3.311     18.998      0.000      56.243      69.565
Estrategia     1.1461      0.768      1.492      0.142      -0.399       2.691
RE             5.2926      1.011      5.233      0.000       3.258       7.327
==============================================================================
Omnibus:                        4.498   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.106   Jarque-Bera (JB):                3.365
Skew:                          -0.546   Prob(JB):                        0.186
Kurtosis:                       3.649   Cond. No.                         32.9
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Modelo 5:
$$PPS = \alpha + \theta RE + \epsilon$$

In [ ]:
x5 = df['RE']
X5 = sm.add_constant(x5)

modelo5 = sm.OLS(df['PPS'], X5).fit()
modelo5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    PPS   R-squared:                       0.406
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     32.75
Date:                Sun, 01 Mar 2026   Prob (F-statistic):           6.64e-07
Time:                        04:43:14   Log-Likelihood:                -118.87
No. Observations:                  50   AIC:                             241.7
Df Residuals:                      48   BIC:                             245.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         62.2122      3.320     18.738      0.000      55.537      68.888
RE             5.6727      0.991      5.722      0.000       3.680       7.666
==============================================================================
Omnibus:                        2.323   Durbin-Watson:                   1.894
Prob(Omnibus):                  0.313   Jarque-Bera (JB):                1.732
Skew:                          -0.454   Prob(JB):                        0.421
Kurtosis:                       3.093   Cond. No.                         32.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Basandonos en los p-values, vemos que ventas y estrategias no explican el precio , entonces si no las incluimos y metemos solo al motor principal "RE", tenemos una $R^2$ aceptable , un poco menor que la del modelo 4 , pero teniendo una variable que si explica el precio, así que em modelo eficiente sería el 5

# Aprendizajes obtenidos <a name="aprendizajes"></a>


El AIC: Aprendí que forzar un modelo lineal sobre datos curvos o multiplicativos genera estimaciones erróneas y que herramientas como la transformación Box-Cox y el uso del AIC son definitivas para seleccionar el modelo que mejor equilibre el ajuste y la complejidad, un ejemplo de aplicación sería predecir el comportamiento de la inflación que suele seguir tendencias multiplicativas, si aplicamos el AIC, los bancos centrales podrían elegir el modelo de proyección más exacto.

Multicolinealidad: Aprendí que mantener variables que explican exactamente distorsionan los cálculos y vuelve inservibles los coeficientes del modelo. En el mundo empresarial, al evaluar por ejemplo el Retorno de Inversión de una campaña publicitaria, los analistas suelen meter métricas redundantes (por ejemplo, "clics en el anuncio" y "visitas a la web") y si no se diagnostica la multicolinealidad, el modelo fallará al asignar qué canal de marketing realmente generó las ventas, llevando a la empresa a desperdiciar su presupuesto en la estrategia equivocada.


In [ ]:
from google.colab import files

files.upload()

Saving Segunda_Evidenciaaa (2).ipynb to Segunda_Evidenciaaa (2).ipynb


{'Segunda_Evidenciaaa (2).ipynb': b'{\n  "nbformat": 4,\n  "nbformat_minor": 0,\n  "metadata": {\n    "colab": {\n      "provenance": []\n    },\n    "kernelspec": {\n      "name": "python3",\n      "display_name": "Python 3"\n    },\n    "language_info": {\n      "name": "python"\n    }\n  },\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "source": [\n        "# Segundo Reporte\\n",\n        "**Periodo:** 16 al 25 de febrero del 2026\\n",\n        "## \xc3\x8dndice\\n",\n        "1. [Objetivo del an\xc3\xa1lisis](#objetivo)\\n",\n        "2. [Simulaci\xc3\xb3n de Modelos e impacto muestral](#simulacion)\\n",\n        "3. [\xc2\xbfQu\xc3\xa9 tranformaci\xc3\xb3n es mejor seg\xc3\xban la m\xc3\xa9trica AIC?](#aic)\\n",\n        "4. [C\xc3\xb3digo en Python y Resultados clave 1.0](#codigo1)\\n",\n        "5. [C\xc3\xb3digo en Python y Resultados clave 2.0](#codigo2)\\n",\n        "6. [Estimamos el modelo original y Revisamos las correlaciones](#correlaciones)\\n",\n        "

In [ ]:
!jupyter nbconvert --to html "Segunda_Evidenciaaa (2).ipynb"
from google.colab import files
files.download("Segunda_Evidenciaaa (2).html")

[NbConvertApp] Converting notebook Segunda_Evidenciaaa (2).ipynb to html
[NbConvertApp] Writing 376583 bytes to Segunda_Evidenciaaa (2).html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>